# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is provided via the following Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

name = metadata.name if hasattr(metadata, 'name') else getattr(metadata, 'name', None)
description = metadata.description if hasattr(metadata, 'description') else getattr(metadata, 'description', None)
print(f"{name}: {description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references.

Here we will print the list of record set `@id`s, then inspect the fields of each record set. All references for record sets and fields must use the `@id`.

In [ ]:
# Helper: print record sets and their fields' @ids
def print_recordsets_and_fields(ds):
    print("Available record sets (referenced by @id):")
    record_set_ids = [rs['@id'] for rs in ds.metadata.to_json().get('recordSet', [])]
    for rs in ds.metadata.to_json().get('recordSet', []):
        print(f"  RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        field_ids = [(f['@id'] if isinstance(f, dict) else str(f)) for f in fields]
        print(f"    Fields: {', '.join(field_ids)}")
    return record_set_ids

record_set_ids = print_recordsets_and_fields(dataset)

# Preview a few records from the first available record set
if record_set_ids:
    print(f"\nPreviewing first 2 records from RecordSet @id: {record_set_ids[0]}")
    for i, x in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(json.dumps(x, indent=2))
        if i >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we use the available RecordSet `@id`s and Field `@id`s obtained in the overview step. Update the variables below with actual `@id` as identified above.

In [ ]:
# Extract data from each record set
# Use only those RecordSet @id's with actual data (as displayed above)
record_sets = record_set_ids
dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"RecordSet {record_set} loaded: {df.shape[0]} records, columns: {df.columns.tolist()}")

# For further steps, pick the first available record set with data
if len(dataframes) > 0:
    primary_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for RecordSet {primary_record_set_id}:\n{dataframes[primary_record_set_id].columns.tolist()}")
    display(dataframes[primary_record_set_id].head())
else:
    print("No record sets with extractable tabular data found.")

## 4. Exploratory Data Analysis (EDA)
Below, we:
- Select a numeric field for analysis (edit to use its `@id`/column name in your dataset),
- Filter records,
- Normalize the numeric field,
- Group by a categorical attribute (if available).

**Replace `<numeric_field_id>` and `<group_field_id>` with field/column `@id`s from your dataset's record set.**

In [ ]:
# Example substitute: set these variable names to column names (@id)
if len(dataframes) > 0:
    df = dataframes[primary_record_set_id]

    # Guess field names by finding possible numeric and categorical fields
    # (For a real use case, inspect df.columns.tolist() and dataset schema)
    candidate_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if candidate_numeric_fields:
        numeric_field = candidate_numeric_fields[0]
        print(f"Using numeric field for demo: {numeric_field}")
    else:
        print("No obvious numeric field found, picking first column.")
        numeric_field = df.columns[0]

    # Pick a candidate group field (string/object type, but not the numeric field)
    candidate_group_fields = [col for col in df.columns if (df[col].dtype == 'object') and (col != numeric_field)]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        print(f"Grouping by: {group_field}")

    # Filtering
    try:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        # Normalization
        filtered_df[numeric_field + "_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())
        # Grouping
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df.head())
    except Exception as e:
        print(f"Error during filtering/normalization/grouping: {e}")
else:
    print("Dataframe not available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here is an example using matplotlib to show the distribution of the selected numeric field and grouped means.

In [ ]:
import matplotlib.pyplot as plt

if len(dataframes) > 0:
    if numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        df[numeric_field].hist(bins=20)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()
    
    if group_field and numeric_field in df.columns:
        plt.figure(figsize=(10,5))
        df.groupby(group_field)[numeric_field].mean().sort_values().plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()

## 6. Conclusion
- Loaded dataset metadata, record set structures, and previewed records using `mlcroissant`.
- Extracted tabular data, performed example filtering and normalization on fields referenced by their `@id`.
- Visualized numeric field distributions and grouped summaries.

To go further: refer to the [mlcroissant documentation](https://github.com/mlcommons/croissant) for support on advanced schema exploration and data integration tasks.